# Solutions — any / all / sorted / min / max / sum and Key Functions

Exercise notebook: `03_pythonic_python/03_builtins_sorting_keys.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · Pipeline run status

**Working solution**

In [ ]:
def pipeline_status(steps):
    if all(s["ok"] for s in steps):
        return "SUCCESS"
    if any(not s["ok"] and s["critical"] for s in steps):
        return "FAILED"
    return "PARTIAL"

**Explanation**

`all()` of an empty iterable is `True`, `any()` is `False` — matches 'nothing failed'. Generators inside short-circuit at the first decisive element.

**Complexity:** O(n) worst case.

**Common mistakes**

- `all([s['ok'] for s in steps])` — the brackets build a full list and lose short-circuiting.

## Exercise 2 — 🟡 Medium · Slowest endpoints

**Working solution**

In [ ]:
from collections import defaultdict

def slowest_endpoints(records, n=3):
    latencies = defaultdict(list)
    for r in records:
        if r.get("path") is not None and r.get("latency_ms") is not None:
            latencies[r["path"]].append(r["latency_ms"])
    averages = [(path, round(sum(v) / len(v), 1)) for path, v in latencies.items()]
    return sorted(averages, key=lambda pa: (-pa[1], pa[0]))[:n]

**Explanation**

Group, aggregate, sort by a composite key, slice. (`heapq.nlargest(n, ...)` is faster for huge k and small n, but needs care with tiebreaks.)

**Complexity:** O(r + k log k).

**Common mistakes**

- Averaging averages across files instead of raw values.

## Exercise 3 — 🟡 Medium · Longest and shortest tenure

**Working solution**

In [ ]:
def tenure_extremes(employees):
    eligible = [e for e in employees if e.get("hire_date") and e.get("status") != "terminated"]
    earliest = min(eligible, key=lambda e: e["hire_date"], default=None)
    latest = max(eligible, key=lambda e: e["hire_date"], default=None)
    return (earliest and earliest["emp_id"], latest and latest["emp_id"])

**Explanation**

`min/max(..., key=..., default=...)` replace the 'best so far' loop and handle empty input. ISO dates sort correctly as strings.

**Complexity:** O(n).

**Common mistakes**

- Sorting the whole list to take the first element (O(n log n)).
- Forgetting `default=` → `ValueError` on empty input.

## Exercise 4 — 🟠 Hard · Competition ranking with ties

**Working solution**

In [ ]:
def rank(scores):
    ordered = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    result = []
    for position, (name, score) in enumerate(ordered, start=1):
        if result and result[-1][2] == score:
            current_rank = result[-1][0]
        else:
            current_rank = position
        result.append((current_rank, name, score))
    return result

**Explanation**

Sort once with a composite key; then `enumerate` gives the 'position' and you only look back one element. This is `RANK()` from SQL window functions.

**Complexity:** O(n log n).

**Common mistakes**

- Using a dict of score → rank (breaks ordering of names).
- Implementing dense rank (`1223`) by mistake.

## Challenge — 🔴 Very Hard · Natural sort for file names and versions

**Working solution**

In [ ]:
import re

_CHUNKS = re.compile(r"(\d+)")

def natural_key(text):
    parts = _CHUNKS.split(text)
    # even indexes are text, odd indexes are digit runs -> tuple of (int|str) in a fixed alternating pattern
    return tuple(int(p) if i % 2 else p.lower() for i, p in enumerate(parts)), text

def natural_sorted(items):
    return sorted(items, key=natural_key)

**Explanation**

`re.split` with a capturing group keeps the separators, so digit runs land on odd indexes. Because every key alternates
str, int, str, int..., comparisons never mix types (which would raise `TypeError` in Python 3). Appending `text` makes ties deterministic.

**Complexity:** O(n log n × len).

**Common mistakes**

- Comparing `int` with `str` → `TypeError: '<' not supported`.
- Lowercasing the tiebreaker too, making order non-deterministic.